# Stage 12C: spatial/typewell cross-fit and K-best lattice

Standalone Colab notebook. It retrains the learned emission under six spatial and five typewell holdouts, then nested-selects deterministic path profiles. It does not create a Kaggle submission. Use a T4 GPU and High-RAM.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import importlib.util, json, os, shutil, subprocess, sys

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
data_dir = drive_root / 'data'
artifact_dir = drive_root / 'artifacts'
assert (data_dir / 'train').is_dir(), f'Missing: {data_dir / "train"}'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
required = ['torch', 'pandas', 'pyarrow', 'sklearn', 'yaml', 'numba']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyarrow', 'scikit-learn', 'pyyaml', 'numba'], check=True)
import torch
assert torch.cuda.is_available(), 'Change the Colab runtime to a T4 GPU and reconnect.'
artifact_dir.mkdir(parents=True, exist_ok=True)
model_env = os.environ.copy()
model_env['PYTHONPATH'] = str(repo_dir / 'src') + ':' + model_env.get('PYTHONPATH', '')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


## Verify or rebuild prerequisites

Existing Drive artifacts are reused. On a fresh Drive this cell rebuilds Stage 11, 11C, 12A, and 12B in order.


In [ ]:
STAGE11_RUN_ID = 'stage11_multicut_delta_u_full_v001'
STAGE11C_RUN_ID = 'stage11c_delta_u_robust_gate_full_v001'
STAGE12A_RUN_ID = 'stage12a_raw_ncc_benchmark_full_v001'
STAGE12B_RUN_ID = 'stage12b_learned_emission_tcn_full_v001'
stage11_run = artifact_dir / STAGE11_RUN_ID
stage11c_run = artifact_dir / STAGE11C_RUN_ID
stage12a_run = artifact_dir / STAGE12A_RUN_ID
stage12b_run = artifact_dir / STAGE12B_RUN_ID
if not (stage11_run / 'fold_coefficient_oof.parquet').is_file():
    subprocess.run(['uv', 'run', 'rogii-delta-u', '--config', 'configs/experiment/stage11_multicut_delta_u.yaml', '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', STAGE11_RUN_ID], cwd=repo_dir, check=True)
if not (stage11c_run / 'gate_summary.json').is_file():
    subprocess.run(['uv', 'run', 'rogii-delta-u-gate', '--config', 'configs/experiment/stage11c_delta_u_robust_gate.yaml', '--stage11-run', str(stage11_run), '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', STAGE11C_RUN_ID], cwd=repo_dir, check=True)
if not (stage12a_run / 'benchmark_summary.json').is_file():
    subprocess.run(['uv', 'run', 'rogii-raw-ncc', '--config', 'configs/experiment/stage12a_raw_ncc_benchmark.yaml', '--stage11-run', str(stage11_run), '--stage11c-run', str(stage11c_run), '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', STAGE12A_RUN_ID], cwd=repo_dir, check=True)
if not (stage12b_run / 'gate_summary.json').is_file():
    subprocess.run([sys.executable, '-m', 'rogii.cli.emission_tcn', '--config', 'configs/experiment/stage12b_learned_emission_tcn.yaml', '--stage11-run', str(stage11_run), '--stage11c-run', str(stage11c_run), '--stage12a-run', str(stage12a_run), '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', STAGE12B_RUN_ID, '--device', 'cuda', '--resume'], cwd=repo_dir, env=model_env, check=True)
assert json.loads((stage12b_run / 'gate_summary.json').read_text())['promoted_to_spatial_emission_audit'] is True
print('Stage 12B gate verified')


## Run Stage 12C

Keep `LIMIT_WELLS = None` for the decision run. This trains eleven new models and may take roughly twice the Stage 12B training time. The standard five models are reused. Rerunning the cell resumes completed spatial/typewell folds.


In [ ]:
RUN_ID = 'stage12c_spatial_kbest_lattice_full_v001'
LIMIT_WELLS = None
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        sys.executable, '-m', 'rogii.cli.emission_lattice',
        '--config', 'configs/experiment/stage12c_spatial_kbest_lattice.yaml',
        '--stage11-run', str(stage11_run), '--stage11c-run', str(stage11c_run),
        '--stage12a-run', str(stage12a_run), '--stage12b-run', str(stage12b_run),
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID, '--device', 'cuda', '--resume',
    ]
    if LIMIT_WELLS is not None:
        command += ['--limit-wells', str(LIMIT_WELLS)]
    log_path = artifact_dir / f'{RUN_ID}_driver.log'
    with log_path.open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(command, cwd=repo_dir, env=model_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end=''); log_handle.write(line); log_handle.flush()
        return_code = process.wait()
    if return_code != 0:
        print('\n'.join(log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-200:]))
        raise RuntimeError(f'Stage 12C failed with exit code {return_code}. Full log: {log_path}')
else:
    print('Reusing completed run:', run_dir)


## Decision summary

The three deltas compare nested path posterior against each family's independently trained expected-offset baseline. K-best is also reconstructed from the outer-fold-selected profiles.


In [ ]:
summary = json.loads((run_dir / 'gate_summary.json').read_text(encoding='utf-8'))
families = summary['family_reports']
decision = {
    'promoted_to_all_train_alignment': summary['promoted_to_all_train_alignment'],
    'standard_base_rmse': families['fold']['base_metrics']['pooled_rmse'],
    'standard_nested_rmse': families['fold']['nested_metrics']['pooled_rmse'],
    'standard_delta': families['fold']['nested_rmse_delta'],
    'spatial_base_rmse': families['spatial_fold']['base_metrics']['pooled_rmse'],
    'spatial_nested_rmse': families['spatial_fold']['nested_metrics']['pooled_rmse'],
    'spatial_delta': families['spatial_fold']['nested_rmse_delta'],
    'typewell_base_rmse': families['typewell_fold']['base_metrics']['pooled_rmse'],
    'typewell_nested_rmse': families['typewell_fold']['nested_metrics']['pooled_rmse'],
    'typewell_delta': families['typewell_fold']['nested_rmse_delta'],
    'bootstrap_95pct': {name: [report['bootstrap']['ci_2_5'], report['bootstrap']['ci_97_5']] for name, report in families.items()},
    'standard_kbest': summary['standard_kbest'],
    'inference_profile': summary['inference_profile'],
    'gates': summary['gates'], 'next_step': summary['next_step'],
}
decision


In [ ]:
import pandas as pd
pd.DataFrame([
    {'family': family, **row}
    for family, report in families.items()
    for row in report['selections']
]).sort_values(['family', 'fold']).reset_index(drop=True)


## Send back

Send both the decision dictionary and the selection table. No submission file is produced.
